<a href="https://colab.research.google.com/github/irlhasnain/RAG-project/blob/main/Hasnain_Khan_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf python-docx nltk gradio pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.5 MB/s eta 0:00:00


In [ ]:
import os
import io
import math
import gc
import json
import textwrap
import re
import random
import collections
from dataclasses import dataclass
from typing import List,Dict,Tuple
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from google.colab import files
from datetime import datetime
from pypdf import PdfReader
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,pipeline
import fitz
import faiss
import numpy as np
import gradio as gr
import torch

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE,torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU"


('cuda', 'Tesla T4')

In [ ]:
MAX_FILES = 5
MAX_MB_PER_FILE = 15 # adust if needed(eg ; 15 MB)
ALLOWED_EXTS = {'.pdf','.docx','.txt'}

print(f"Please upload up to {MAX_FILES} files (PDF/DOCX/TXT).MAX file size: {MAX_MB_PER_FILE} MB")
uploaded = files.upload() #user selects files

Please upload up to 5 files (PDF/DOCX/TXT).MAX file size: 15 MB


Saving Nazia_Khan_Resume.pdf to Nazia_Khan_Resume.pdf
Saving N2443006042 (5).pdf to N2443006042 (5).pdf
Saving Hasnain_Khan_Resume_DataScience.pdf to Hasnain_Khan_Resume_DataScience.pdf
Saving certificate-7y3c2dcmu7qc-1785500840.pdf to certificate-7y3c2dcmu7qc-1785500840.pdf
Saving 1.8_Diligence_Summary_16x9.pdf to 1.8_Diligence_Summary_16x9.pdf


In [ ]:
def ext_of(name):
    return os.path.splitext(name)[1].lower()

assert len(uploaded) > 0, "No files uploaded."
assert len(uploaded) <= MAX_FILES, f"Please upload at most {MAX_FILES} files."

docs_raw = []

for fname, b in uploaded.items():
    assert ext_of(fname) in ALLOWED_EXTS, (
        f"Unsupported file type for {fname}. Allowed: {ALLOWED_EXTS}"
    )

    size_mb = len(b) / (1024 * 1024)
    assert size_mb <= MAX_MB_PER_FILE, (
        f"{fname} is {size_mb:.2f} MB, exceeds {MAX_MB_PER_FILE} MB."
    )

    docs_raw.append((fname, b))

print("Uploaded files:", [d[0] for d in docs_raw])

Uploaded files: ['Nazia_Khan_Resume.pdf', 'N2443006042 (5).pdf', 'Hasnain_Khan_Resume_DataScience.pdf', 'certificate-7y3c2dcmu7qc-1785500840.pdf', '1.8_Diligence_Summary_16x9.pdf']


In [ ]:
def read_txt(bytes_blob: bytes) -> str:
  return io.BytesIO(bytes_blob).read().decode('utf - 8', errors = 'ignore')

In [ ]:
def read_pdf(bytes_blob: bytes) -> str:
    reader = PdfReader(io.BytesIO(bytes_blob))
    texts = []
    for page in reader.pages:
        try:
            texts.append(page.extract_text() or "")
        except:
            texts.append("")
    return "\n".join(texts)

In [ ]:
def read_docx(bytes_blob) -> str:
    fh = io.BytesIO(bytes_blob)
    doc = DocxDocument(fh)
    return "\n".join([p.text for p in doc.paragraphs])

In [ ]:
def load_text_by_ext(fname:str, blob: bytes) -> str:
    ext = os.path.splitext(fname)[1].lower()
    if ext == '.txt':
        return read_txt(blob)
    elif ext == '.pdf':
        return read_pdf(blob)
    elif ext == '.docx':
        return read_docx(blob)
    else:
        raise ValueError(f"Unsupported extension: {ext}")

In [ ]:
# Remove unwanted spaces
docx_text = []
for fname,blob in docs_raw:
  text = load_text_by_ext(fname,blob)
  #light cleanup
  text = re.sub(r's+\n','\n',text)
  text = re.sub(r'\n{3,}','\n\n',text)
  text = text.strip()
  docx_text.append({"name":fname,"text":text})

for d in docx_text:
    print(d["name"],"characters:", len(d["text"]))

Nazia_Khan_Resume.pdf characters: 3278
N2443006042 (5).pdf characters: 3767
Hasnain_Khan_Resume_DataScience.pdf characters: 5167
certificate-7y3c2dcmu7qc-1785500840.pdf characters: 12
1.8_Diligence_Summary_16x9.pdf characters: 1383


In [ ]:
def _clean_text(s: str) ->str:
  s= re.sub(r'\s+',' ',s).strip()
  s= re.sub(r'^\W+|\W+$',' ',s).strip()
  return s


To clean and normalize expected pdf text before using it for the further processing this helps produce clean and consistent text

In [ ]:
def extract_title_from_pdf_layout(pdf_bytes: bytes, consider_pages=3, top_band=0.35):
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")

    if doc.page_count == 0:
        return None

    spans, page_heights = [], {}

    for pno in range(min(consider_pages, doc.page_count)):
        pg = doc.load_page(pno)  # pno means page number
        page_heights[pno] = pg.rect.height

        data = pg.get_text("dict")

        for b in data.get("blocks", []):
            for l in b.get("lines", []):
                for s in l.get("spans", []):
                    txt = _clean_text(s.get("text", ""))

                    if not txt or len(txt) < 3:
                        continue

                    size = float(s.get("size", 0))
                    font = (s.get("font") or "").lower()
                    is_bold = ("bold" in font) or ("black" in font)
                    y0 = s.get("bbox", [0, 0, 0, 0])[1]

                    spans.append({
                        "page": pno,
                        "text": txt,
                        "size": size,
                        "bold": is_bold,
                        "y0": y0
                    })

    if not spans:
        return None

    # keep top-of-page band only
    top_spans = [
        sp for sp in spans
        if page_heights.get(sp["page"], 1)
        and (sp["y0"] / page_heights[sp["page"]]) <= top_band
    ]

    spans = top_spans or spans

    # drop strings repeating on >= 2 pages (likely header)
    per_text_pages = collections.defaultdict(set)

    for sp in spans:
        per_text_pages[sp["text"]].add(sp["page"])

    spans = [
        sp for sp in spans
        if not (
            len(per_text_pages[sp["text"]]) >= 2
            and len(sp["text"]) <= 80
        )
    ] or spans

    # groups consecutive lines with same size+bold on same page
    # to capture multi-line titles
    groups = []

    spans.sort(key=lambda x: (x["page"], -x["size"], x["y0"]))

    for sp in spans:
        placed = False

        for g in groups:
            if (
                g["page"] == sp["page"]
                and abs(g["size"] - sp["size"]) < 0.5
                and g["bold"] == sp["bold"]
            ):
                if abs(sp["y0"] - g["lines"][-1][1]) < 22:
                    g["lines"].append((sp["text"], sp["y0"]))
                    placed = True
                    break

        if not placed:
            groups.append({
                "page": sp["page"],
                "size": sp["size"],
                "bold": sp["bold"],
                "lines": [(sp["text"], sp["y0"])]
            })

    groups.sort(
        key=lambda g: (
            g["size"],
            g["bold"],
            -len(g["lines"]),
            -1.0 / min([y for _, y in g["lines"]] or [1])
        ),
        reverse=True
    )

    for g in groups:
        lines_sorted = [
            t for t, _ in sorted(
                g["lines"],
                key=lambda x: x[1]
            )
        ]

        cand = _clean_text(" ".join(lines_sorted))

        if 5 <= len(cand) <= 200:
            return cand

    best = max(
        spans,
        key=lambda sp: (sp["size"], sp["bold"], -sp["y0"])
    )

    return best["text"] if best else None

In [ ]:
def extract_title_from_docx(docx_bytes: bytes):

    d = DocxDocument(io.BytesIO(docx_bytes))

    def paragraph_score(p):

        text = _clean_text(p.text)

        if not text or len(text) < 3:
            return None

        style_name = (p.style.name if p.style else "").lower()

        max_size_pt, any_bold = 0.0, False

        for run in p.runs:

            if run.font is not None:

                if run.font.size:

                    try:
                        max_size_pt = max(
                            max_size_pt,
                            float(run.font.size.pt)
                        )
                    except:
                        pass

                if run.font.bold:
                    any_bold = True

        style_priority = 0

        if "title" in style_name:
            style_priority = 3

        elif "heading 1" in style_name or style_name == "heading1":
            style_priority = 2

        elif "heading" in style_name:
            style_priority = 1

        return {
            "text": text,
            "style_priority": style_priority,
            "size": max_size_pt,
            "bold": any_bold
        }

    candidates = []

    for p in d.paragraphs:

        sc = paragraph_score(p)

        if sc:
            candidates.append(sc)

    if not candidates:
        return None

    candidates.sort(
        key=lambda c: (
            c["style_priority"],
            c["size"],
            c["bold"],
            -len(c["text"])
        ),
        reverse=True
    )

    return candidates[0]["text"]

This code automatically detets the title of a word document by looking at paragraph styles, font size

In [ ]:
def extract_title_from_txt_firstline(txt: str):
  for line in txt.splitlines():
    if line.strip(): return _clean_text(line)
  return None

In [ ]:
name_to_title = {}
name_to_blob = {fname: blob for fname, blob in docs_raw}
for item in docx_text:
  fname = item["name"]; blob = name_to_blob[fname]
  ext = os.path.splitext(fname) [1].lower()
  title = None
  try:
    if ext == ".pdf":
      title = extract_title_from_pdf_layout(blob)
    elif ext == ".docx":
      title = extract_title_from_docx(blob)
    elif ext == ".txt":
      title = extract_title_from_txt_firstline(item["text"])
  except Exception:
    title = None
  if not title: title = os.path.splitext(fname) [0] # final fallback
  name_to_title[fname] = title
print("Detected titles:")
for d in docs_text:
  print(f". {d['name']} {name_to_title[d['name']]}")


Detected titles:
. Nazia_Khan_Resume.pdf NAZIA KHAN
. N2443006042 (5).pdf MADHYA PRADESH MADHYA KSHETRA VIDYUT VITRAN COMPANY LTD
. Hasnain_Khan_Resume_DataScience.pdf HASNAIN KHAN
. certificate-7y3c2dcmu7qc-1785500840.pdf Hasnain Khan
. 1.8_Diligence_Summary_16x9.pdf clR clR clR


In [ ]:
def chunk_text(text, target_chars=1400, overlap_chars=200):

    if not text or not text.strip():
        return []

    sentences = text.split(".")
    chunks = []
    buf = ""

    for s in sentences:

        s = s.strip()

        if not s:
            continue

        if len(buf) + len(s) + 1 <= target_chars:
            buf = (buf + " " + s).strip()

        else:
            if buf:
                chunks.append(buf.strip())

            # overlap tail
            tail = buf[-overlap_chars:]

            buf = (tail + " " + s).strip()

    if buf:
        chunks.append(buf.strip())

    return chunks

In [ ]:
all_docs_chunks = []

for d in docx_text:

    ch = chunk_text(
        d["text"],
        target_chars=1400,
        overlap_chars=200
    )

    all_docs_chunks.append({
        "name": d["name"],
        "chunks": ch
    })

    print(d["name"], "->", len(ch), "chunks")

Nazia_Khan_Resume.pdf -> 3 chunks
N2443006042 (5).pdf -> 4 chunks
Hasnain_Khan_Resume_DataScience.pdf -> 5 chunks
certificate-7y3c2dcmu7qc-1785500840.pdf -> 1 chunks
1.8_Diligence_Summary_16x9.pdf -> 1 chunks


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)

# Flatten chunks + keep metadata
corpus_texts, corpus_meta = [], []

# meta: (doc_id, doc_name, chunk_id)
for doc_id, d in enumerate(all_docs_chunks):

    for chunk_id, c in enumerate(d["chunks"]):

        corpus_texts.append(c)

        corpus_meta.append(
            (doc_id, d["name"], chunk_id)
        )


# Compute embeddings
BATCH = 64

embs = []

for i in range(0, len(corpus_texts), BATCH):

    embs.extend(
        embedder.encode(
            corpus_texts[i:i + BATCH],
            normalize_embeddings=True,
            show_progress_bar=False
        )
    )


embs = np.vstack(embs).astype("float32")


# FAISS index
# Cosine similarity -> IndexFlatIP with normalized vectors
index = faiss.IndexFlatIP(embs.shape[1])

index.add(embs)

print("Index built with", index.ntotal, "vectors.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built with 14 vectors.


In [ ]:
import torch
from transformers import pipeline


# ============================================================
# DEVICE
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

device_id = 0 if DEVICE == "cuda" else -1

print("Device:", DEVICE)


# ============================================================
# GENERATION MODEL
# ============================================================

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    task="text-generation",
    model=GEN_MODEL,
    device=device_id
)

print("Generation model loaded successfully.")
print("Model:", GEN_MODEL)

Device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Generation model loaded successfully.
Model: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
def safe_truncate(txt, max_chars = 2400):
  #keep a little room
  return txt if len(txt)<= max_chars else txt[:max_chars]+"..."

In [ ]:
# ------------------------------------------------------------
# Improved summarization: fact-rich bullet points
# Fast version - limited number of LLM calls
# ------------------------------------------------------------

def summarize_long_text(
    text: str,
    chunk_chars: int = 2500,
    overlap: int = 200,
    max_chunks: int = 6
) -> str:

    # ========================================================
    # 1. Split document into chunks
    # ========================================================

    parts = chunk_text(
        text,
        target_chars=chunk_chars,
        overlap_chars=overlap
    )

    if not parts:
        return "- No content available."


    # ========================================================
    # 2. Select representative chunks
    # ========================================================

    if len(parts) > max_chunks:

        indices = np.linspace(
            0,
            len(parts) - 1,
            max_chunks,
            dtype=int
        )

        selected_parts = [
            parts[i]
            for i in indices
        ]

    else:

        selected_parts = parts


    # ========================================================
    # 3. Combine selected content
    # ========================================================

    selected_text = "\n\n".join(
        selected_parts
    )

    selected_text = safe_truncate(
        selected_text,
        6000
    )


    # ========================================================
    # 4. ONE LLM CALL
    # ========================================================

    prompt = f"""
You are a document summarization assistant.

Summarize the following document content into
7-10 concise, specific and factual bullet points.

Preserve important:
- rules
- procedures
- requirements
- people
- organizations
- places
- dates
- years
- numbers
- findings
- outcomes

Do not invent information.

Return ONLY bullet points.
Each bullet must start with "-".

DOCUMENT:
{selected_text}

SUMMARY:
"""

    try:

        output = generator(
            prompt,
            max_new_tokens=300,
            do_sample=False,
            return_full_text=False,
            clean_up_tokenization_spaces=False
        )

        final = output[0][
            "generated_text"
        ].strip()

    except Exception as e:

        print(
            "Warning during summarization:",
            e
        )

        return "- Unable to generate summary."


    # ========================================================
    # 5. Clean bullet points
    # ========================================================

    lines = [
        line.strip()
        for line in final.splitlines()
        if line.strip()
    ]

    bullets = []

    for line in lines:

        if not line.startswith("-"):

            line = "- " + line.lstrip(
                ".*- "
            )

        bullets.append(line)


    # ========================================================
    # 6. Fallback
    # ========================================================

    if not bullets:

        return "- " + final


    return "\n".join(
        bullets[:10]
    ).strip()

In [ ]:
# Re-run to rebuild summaries with the new style

doc_summaries = {}

for d in docx_text:

    fname = d["name"]

    print(
        f"Summarizing: {name_to_title[fname]}"
    )

    doc_summaries[fname] = summarize_long_text(
        d["text"]
    )

print("Done.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: NAZIA KHAN


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: MADHYA PRADESH MADHYA KSHETRA VIDYUT VITRAN COMPANY LTD


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: HASNAIN KHAN


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: Hasnain Khan


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing: clR clR clR
Done.


In [ ]:
def search (query: str,top_k: int= 6)-> List[Tuple[float,str,Tuple[int,str,int]]]:
  qe = embedder.encode([query],normalize_embeddings =True)
  D, I = index.search(qe.astype('float32'),top_k)
  hits = []
  for score, idx in zip(D[0],I[0]):
    hits.append((float(score),corpus_texts[idx],corpus_meta[idx])) #score, chunk_text,(doc_id,doc_name,chunk_id))
  return hits

In [ ]:
#== Recovery CELL - run this once before FAQ generation ===

# ensure ANSWER_PROMPt_TMPL exists

if "ANSWER_PROMPT_TMPL" not in globals():
  ANSWER_PROMPT_TMPL = """You arre a helpful assistant. Answer the question Strictly using the provided content
  IF the answer is not in the context, say "I don't know from the provided documents."

  Context:
  {context}

  Question: {question}

  Answer:"""

  #ensure search() exists

if "search" not in globals():
    raise RuntimeError(
      "`search()` is missing. You MUST re-run the FAISS + embeddings cell that built the index and defined the `search()` function."
  )
print("ANSWER_PROMPT_TMPL restored.`search(` presence =",'search' in globals())

ANSWER_PROMPT_TMPL restored.`search(` presence = True


In [ ]:
def safe_truncate(txt, max_chars=2400):
    # keep a little room
    return txt if len(txt) <= max_chars else txt[:max_chars] + "_"


FAQ_QS_PROMPT = """
You are a helpful assistant that generates a list of 12 highly specific and detailed questions based on the provided document content. The questions should cover the key facts, procedures, and entities mentioned. Avoid generic questions.

DOCUMENT:
{doc}

QUESTIONS:
- """

In [ ]:
FAQ_QS_PROMPT = """
You are a helpful assistant that generates a list of 12 highly specific and detailed questions based on the provided document content. The questions should cover the key facts, procedures, and entities mentioned. Avoid generic questions.

DOCUMENT:
{doc}

QUESTIONS:
- """

In [ ]:
# === Improved FAQ Generation v2 (varied questions, grounded answers, no placeholders) ===

# --- helpers to detect/clean generic questions ---
GENERIC_PATTERNS = [
    r"\bmain point\b",
    r"\bwhy .* important\b",
    r"\bintended audience\b",
    r"\bscope\b",
    r"\bconclusion\b",
    r"\boverview\b",
    r"\bsum(mary|marize)\b",
    r"\badditional detail\b",
    r"\bwhat is .*document\b",
    r"\bwhat does the document\b",
]

GENERIC_RE = re.compile("|".join(GENERIC_PATTERNS), re.I)

In [ ]:
def looks_specific(q: str) -> bool:
    q = q.strip().rstrip("?")

    if len(q.split()) < 6:  # avoid very short questions
        return False

    if GENERIC_RE.search(q):
        return False

    # discourage ultra-vague starts
    if re.match(r"(?i)^(what is|what does|why is|why are|who is)", q[:20]):
        return False

    return True


# Step 26: Remove duplicate items while keeping the original order
def dedup_preserve_order(items):
    seen = set()
    out = []

    for item in items:
        key = item.strip().lower()

        if key not in seen:
            seen.add(key)
            out.append(item)

    return out

In [ ]:
def dedup_preserve_order(items):
    seen, out = set(), []

    for x in items:
        key = re.sub(r"\s+", " ", x.strip().lower())

        if key in seen:
            continue

        seen.add(key)
        out.append(x.strip())

    return out

In [ ]:
def generate_varied_questions_from_summary(
    summary_text: str,
    k: int = 12
) -> list:
    base = safe_truncate(summary_text, 4500)
    prompt = FAQ_QS_PROMPT.format(doc=base)

    out = generator(
        prompt,
        max_new_tokens=256,
        temperature=0.9,  # sampling for variety
        top_p=0.9,
        do_sample=True
    )[0]["generated_text"]

    qs = [
        q.strip().rstrip("?") + "?"
        for q in out.splitlines()
        if q.strip()
    ]

    qs = dedup_preserve_order(qs)

    # filter generic
    qs = [q for q in qs if looks_specific(q)]

    # keep top ~10 and randomize a bit to avoid same 5 every run
    random.shuffle(qs)

    return qs[:max(k, 5)]

In [ ]:
# --- Answering with robust context (doc retrieval + summary fallback) ---
def answer_with_doc_robust(question: str, doc_name: str, summary_text: str) -> str:
    # primary: restrict hits to this doc
    hits = search(question, top_k=12)
    hits = [h for h in hits if h[2][1] == doc_name]

    ctx_blocks, seen = [], set()

    for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
        if len("\n".join(ctx_blocks)) > 4200:
            break

        sig = hash(txt)

        if sig in seen:
            continue

        seen.add(sig)
        ctx_blocks.append(txt)

    # fallback: if retrieval is weak, include summary
    context = "\n\n".join(ctx_blocks)

    if len(context) < 400 and summary_text:
        # prepend summary to ensure we have something factual
        context = (
            summary_text.strip()
            + "\n\n"
            + context.strip()
        ).strip()

    if not context:
        # absolute fallback to summary only (better than "Not specified")
        context = summary_text if summary_text else "No context."

    prompt = ANSWER_PROMPT_TMPL.format(
        context=context,
        question=question
    )

    ans = generator(
        prompt,
        max_new_tokens=220,
        temperature=0.2,       # deterministic-ish answers
        do_sample=False
    )[0]["generated_text"].strip()

    # last resort: if model still returns something empty or evasive,
    # try a 2nd pass using only summary
    if (not ans) or len(ans.split()) < 3 or "I don't know" in ans:
        if summary_text:
            prompt2 = ANSWER_PROMPT_TMPL.format(
                context=summary_text,
                question=question
            )

            ans2 = generator(
                prompt2,
                max_new_tokens=200,
                temperature=0.2,
                do_sample=False
            )[0]["generated_text"].strip()

            if ans2 and "I don't know" not in ans2:
                ans = ans2

    # keep it tight
    return ans

In [ ]:
def generate_faqs_for_document_v2(
    doc_name: str,
    title: str,
    text: str,
    summary: str
) -> list:

    # ensure we have a seed summary; if not, synthesize a short one
    seed = summary if (summary and len(summary) > 60) else summarize_long_text(text)

    # 1) make 12 candidates from summary
    candidates = generate_varied_questions_from_summary(
        seed,
        k=12
    )

    # 2) if we somehow have <5, synthesize from top bullets
    if len(candidates) < 5:

        bullets = [
            ln[2:].strip()
            for ln in seed.splitlines()
            if ln.strip().startswith("- ")
        ]

        for b in bullets:

            if len(candidates) >= 5:
                break

            frag = " ".join(b.split()[:12])

            candidates.append(
                f"What specific findings does the document report about {frag}?"
            )

        candidates = dedup_preserve_order(candidates)

    # 3) pick 5 best-looking questions
    final_qs = candidates[:5]

    # 4) answer each question robustly
    faqs = []

    for q in final_qs:
        a = answer_with_doc_robust( # Corrected function call
            q,
            doc_name,
            summary
        )

        faqs.append((q, a))

    return faqs

In [ ]:
# --- Run for all docs (uses previously computed:
# docs_text, name_to_title, doc_summaries) ---

doc_faqs = {}

for d in docx_text:

    fname = d["name"]

    print(
        "Generating FAQs for",
        name_to_title[fname]
    )

    doc_faqs[fname] = generate_faqs_for_document_v2(
        fname,
        name_to_title[fname],
        d["text"],
        doc_summaries.get(fname)
    )

print("Done (Improved FAQ Generation v2).")

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating FAQs for NAZIA KHAN


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to

Generating FAQs for MADHYA PRADESH MADHYA KSHETRA VIDYUT VITRAN COMPANY LTD


[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

...

Generating FAQs for Hasnain Khan


[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generating FAQs for clR clR clR


[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Done (Improved FAQ Generation v2).


In [ ]:
def show_document_overview():
  for d in docx_text:
    name = d["name"]
    print("=" * 100)
    print(name)
    print("- Summary -")
    print(textwrap.fill(doc_summaries[name], width=100))
    print("\n- FAQs")
    for i, (q, a) in enumerate(doc_faqs[name], 1):
      print(f"Q{i}: {q}\n{i}: {a}\n")
show_document_overview()

Nazia_Khan_Resume.pdf
- Summary -
- Naza Khan is an experienced Data Analyst with a strong analytical background. She has completed a
Master of Science in Computer Science (MSc CS) from Panshten University, Shahdol, Pakistan, and a
Bachelor of Science in Mathematics (BSc Math) from Panshten University, Shahdol, Pakistan. She has
also completed a Data Analytics course in 2025-2026. - Khan specializes in using Python, SQL, and
advanced Excel for data cleaning, exploration, and visualization. She has experience in EDA, BI
tools, and data engineering. - Her projects include household power consumption analysis, automobile
data cleaning and visualization, and UPI transactions analysis. She uses Python libraries such as
NumPy, Pandas, Matplotlib, Seaborn, and Power BI for her work. - Khan is proficient in multiple
programming languages and can perform complex data analysis tasks using these languages. She has
extensive knowledge of statistical methods and can apply them effectively in her wo

In [ ]:
# Build mapping: title -> filename, and "All"
TITLE_TO_NAME = {name_to_title[d["name"]]: d["name"] for d in docx_text}
DOC_TITLES = ["All"] + list(TITLE_TO_NAME.keys())

def qa_interface(question, title_scope):
    # map chosen title to original filename
    if title_scope != "All":
        doc_scope = TITLE_TO_NAME[title_scope]
        # Restrict to that doc
        hits = search(question, top_k=12)
        hits = [h for h in hits if h[2][1] == doc_scope]
        ctx = []
        seen = set()
        for sc, txt, meta in sorted(hits, key=lambda x: -x[0]):
            if len(" ".join(ctx)) > 4200: break
            sig = hash(txt)
            if sig in seen: continue
            seen.add(sig); ctx.append(txt)
        context = "\n\n".join(ctx) if ctx else "No context available."
        prompt = ANSWER_PROMPT_TMPL.format(context=context, question=question)
        out = generator(prompt, max_new_tokens=256, temperature=0.2)[0]["generated_text"].strip()
        sources = [{"title": name_to_title[m[1]], "chunk_id": m[2], "score": float(s)} for (s,_,m) in hits[:5]]
        return out, json.dumps(sources, indent=2)
    else:
        res = answer_question(question, top_k=8)
        # decorate sources with titles
        for s in res["sources"]:
            s["title"] = name_to_title.get(s["doc_name"], s["doc_name"])
            del s["doc_name"]
        return res["answer"], json.dumps(res["sources"], indent=2)

In [ ]:
def get_summary(title_scope):
    if title_scope == "All":
        merged = []
        for d in docs_text:
            fname = d["name"]
            merged.append(f"### {name_to_title[fname]}\n{doc_summaries[fname]}")
        return "\n\n".join(merged)
    fname = TITLE_TO_NAME[title_scope]
    return doc_summaries.get(fname, "No summary available.")

In [ ]:
def get_faqs(title_scope):
    if title_scope == "All":
        out = []
        for d in docs_text:
            fname = d["name"]
            friendly = name_to_title[fname]
            out.append("### " + friendly + "\n" + "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(doc_faqs[fname])]))
        return "\n\n".join(out)
    fname = TITLE_TO_NAME[title_scope]
    faqs = doc_faqs.get(fname, [])
    return "\n".join([f"Q{i+1}: {q}\nA{i+1}: {a}" for i,(q,a) in enumerate(faqs)]) or "No FAQs."

In [ ]:
with gr.Blocks(title="Mini NotebookLM (Local RAG)") as demo:
    gr.Markdown("#  Mini NotebookLM (Local RAG) - Colab\nUpload -> Summaries & FAQs -> Ask Questions, grounded in...")
    with gr.Row():
        title_scope = gr.Dropdown(DOC_TITLES, value="All", label="Document Scope (by Title)")
    with gr.Tab("Ask Questions"):
        question = gr.Textbox(label="Your question", placeholder="Ask something grounded in your uploaded docs...")
        btn = gr.Button("Answer")
        answer = gr.Textbox(label="Answer", lines=8)
        sources = gr.Textbox(label="Top Source Chunks (title, chunk_id, score)", lines=8)
        btn.click(qa_interface, inputs=[question, title_scope], outputs=[answer, sources])
    with gr.Tab("Summaries"):
        btn_sum = gr.Button("Show Summary")
        sum_box = gr.Textbox(label="Summary", lines=20)
        btn_sum.click(get_summary, inputs=[title_scope], outputs=[sum_box])
    with gr.Tab("FAQs"):
        btn_faq = gr.Button("Show FAQs (5)")
        faq_box = gr.Textbox(label="FAQs", lines=20)
        btn_faq.click(get_faqs, inputs=[title_scope], outputs=[faq_box])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebfc85af2979c7cb95.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
